---

<div style="border:1px solid #ddd; border-radius:12px; padding:20px; box-shadow:0px 4px 10px rgba(0,0,0,0.1);">

<h1 style="color:#4facfe;">📊 Data Analytics : Superstore Sales</h1>

<p style="font-size:15px;">
Exploration, nettoyage et visualisation des données du <a href="https://www.kaggle.com/datasets/vivek468/superstore-dataset-final">dataset Superstore</a>.
</p>

<hr>

<p>
👤 <b>Kodjo Jean DEGBEVI</b><br>
📘 <a href="../README.md">Readme</a>
</p>
<p style="font-style: italic;">Copyright (c) 2026 Kodjo Jean DEGBEVI. Distribué sous licence CC BY-NC-SA 4.0.</p>

</div>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import os
import sys
from pathlib import Path

CURRENT_DIR = os.getcwd()
ROOT_DIR = Path(CURRENT_DIR).parent
sys.path.append(str(ROOT_DIR))

from src.utils import get_us_state_abbrev

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style="whitegrid")

## Chargement et Compréhension

**Contexte**

Face à une demande croissante et une concurrence féroce, une grande enseigne
de distribution américaine sollicite une expertise analytique pour identifier
ses stratégies les plus performantes sur la période **2014–2017**.

---

**Objectif**

Déterminer les **produits, régions, catégories et segments de clientèle**
à cibler ou à éviter — et formuler des recommandations concrètes et actionnables.

---

**Questions directrices**

| # | Question | Angle |
|---|---|---|
| Q1 | Quelle est la santé financière globale de l'entreprise ? | Descriptif |
| Q2 | Quelles régions et états sont rentables ou déficitaires ? | Diagnostique |
| Q3 | Quelles catégories et sous-catégories créent ou détruisent de la valeur ? | Diagnostique |
| Q4 | Les remises sont-elles une stratégie rentable ou un problème structurel ? | Diagnostique |
| Q5 | Quel profil client est le plus rentable, et pourquoi ? | Prescriptif |

---

### **01**

In [2]:
df = pd.read_csv("../data/raw/Sample - Superstore.csv", encoding='latin1')
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   str    
 2   Order Date     9994 non-null   str    
 3   Ship Date      9994 non-null   str    
 4   Ship Mode      9994 non-null   str    
 5   Customer ID    9994 non-null   str    
 6   Customer Name  9994 non-null   str    
 7   Segment        9994 non-null   str    
 8   Country        9994 non-null   str    
 9   City           9994 non-null   str    
 10  State          9994 non-null   str    
 11  Postal Code    9994 non-null   int64  
 12  Region         9994 non-null   str    
 13  Product ID     9994 non-null   str    
 14  Category       9994 non-null   str    
 15  Sub-Category   9994 non-null   str    
 16  Product Name   9994 non-null   str    
 17  Sales          9994 non-null   float64
 18  Quantity       9994

In [3]:
df.drop(columns=['Row ID', 'Postal Code']).describe(include='all')

,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
count,9994,9994,9994,9994,9994,9994,9994,9994,9994,9994,9994,9994,9994,9994,9994,9994.00,9994.00,9994.00,9994.00
unique,5009,1237,1334,4,793,793,3,1,531,49,4,1862,3,17,1850,NaN,NaN,NaN,NaN
top,CA-2017-100111,9/5/2016,12/16/2015,Standard Class,WB-21850,William Brown,Consumer,United States,New York City,California,West,OFF-PA-10001970,Office Supplies,Binders,Staple envelope,NaN,NaN,NaN,NaN
freq,14,38,35,5968,37,37,5191,9994,915,2001,3203,19,6026,1523,48,NaN,NaN,NaN,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,229.86,3.79,0.16,28.66
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,623.25,2.23,0.21,234.26
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.44,1.00,0.00,-6599.98
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17.28,2.00,0.00,1.73
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,54.49,3.00,0.20,8.67
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,209.94,5.00,0.20,29.36


In [4]:
print(f"Lignes dupliquées : {df.duplicated().sum()}")

Lignes dupliquées : 0


**Aucune donnée manquante**

**Aucun doublons**

### **02** : dictionnaire des données

| Colonne        | Type            | Ce que c'est vraiment |
|----------------|-----------------|----------------------|
| Row ID         | Identifiant     | Numéro de ligne |
| Order ID       | Identifiant     | Identifiant unique d'une commande — une commande peut avoir plusieurs lignes |
| Order Date     | Date            | Date à laquelle le client a passé la commande |
| Ship Date      | Date            | Date à laquelle la commande a été expédiée |
| Ship Mode      | Catégorie       | Mode de livraison choisi : Same Day / First Class / Second Class / Standard Class |
| Customer ID    | Identifiant     | Identifiant unique du client |
| Customer Name  | Texte           | Nom du client |
| Segment        | Catégorie       | Consumer, Corporate, Home Office — trois comportements d'achat très différents avec des cycles de décision, des volumes et des marges différents |
| Country        | Géographie      | Pays — ici toujours "United States", colonne sans variance |
| City           | Géographie      | Ville de livraison |
| State          | Géographie      | État américain — 49 mentionnés ici |
| Postal Code    | Géographie      | Code postal |
| Region         | Géographie      | Région agrégée : West / East / Central / South |
| Product ID     | Identifiant     | Identifiant unique du produit |
| Category       | Catégorie       | Famille produit : Furniture / Office Supplies / Technology |
| Sub-Category   | Catégorie       | Sous-famille : 17 valeurs (Chairs, Phones, Binders…) |
| Product Name   | Texte           | Nom exact du produit |
| Sales          | Mesure         | Chiffre d'affaires de la ligne — prix unitaire × quantité après remise |
| Quantity       | Mesure         | Nombre d'unités vendues |
| Discount       | Mesure         | Taux de remise appliqué — de 0 à 0.8 (0% à 80%) |
| Profit         | Mesure         | Bénéfice net de la ligne — après coûts et remise |

## Feature engineering & Nettoyage

### 00

In [5]:
df.sample(5)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
1542,1543,CA-2014-166884,3/11/2014,3/16/2014,Second Class,CK-12205,Chloris Kastensmidt,Consumer,United States,Columbus,Ohio,43229,East,OFF-FA-10001561,Office Supplies,Fasteners,Stockwell Push Pins,10.46,6,0.20,1.70
5554,5555,US-2014-159618,11/12/2014,11/16/2014,Standard Class,DB-12970,Darren Budd,Corporate,United States,Houston,Texas,77036,Central,FUR-BO-10004467,Furniture,Bookcases,Bestar Classic Bookcase,67.99,1,0.32,-13.00
9123,9124,US-2016-158288,11/26/2016,11/28/2016,Second Class,EH-13945,Eric Hoffmann,Consumer,United States,Philadelphia,Pennsylvania,19120,East,OFF-BI-10003364,Office Supplies,Binders,Binding Machine Supplies,78.76,9,0.70,-57.76
3023,3024,CA-2017-124674,11/17/2017,11/23/2017,Standard Class,JB-16000,Joy Bell-,Consumer,United States,Brownsville,Texas,78521,Central,FUR-BO-10002202,Furniture,Bookcases,"Atlantic Metals Mobile 2-Shelf Bookcases, Cust...",327.73,2,0.32,-14.46
5239,5240,CA-2016-116561,9/11/2016,9/17/2016,Standard Class,EB-14110,Eugene Barchas,Consumer,United States,San Jose,California,95123,West,OFF-ST-10004186,Office Supplies,Storage,"Stur-D-Stor Shelving, Vertical 5-Shelf: 72""H x...",332.94,3,0.00,6.66


**Rien à faire sur les identifiant**

1. Je convertis les dates au bon format
2. Je supprime la colonne `Country` car il est invariant
3. Je vais produire les features suivantes : 
   - *`Profit Margin`* =  *`Profit` / `Sales`* : pour voir la rentabilité réelle (marge bénéficiaire) en %
   - *`Ship Delay`* = *`Ship Date` - `Order Date`*
   - *`Year`, `Month`, `Trimestre`* extraits de `Order Date` pour les analyses de la saisonnalité et la croissance

### **01**

In [6]:
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date']  = pd.to_datetime(df['Ship Date'])
df.drop(columns=['Country'], inplace=True)

In [7]:
df['Profit Margin'] = df['Profit'] / df['Sales']
df['Ship Delay'] = (df['Ship Date'] - df['Order Date']).dt.days
df['Order Year'] = df['Order Date'].dt.year
df['Order Month'] = df['Order Date'].dt.month
df['Order Day'] = df['Order Date'].dt.day

### **02**

In [8]:
df.to_csv("../data/processed/superstore_processed.csv", index=False)

### **03**

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 25 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Row ID         9994 non-null   int64         
 1   Order ID       9994 non-null   str           
 2   Order Date     9994 non-null   datetime64[us]
 3   Ship Date      9994 non-null   datetime64[us]
 4   Ship Mode      9994 non-null   str           
 5   Customer ID    9994 non-null   str           
 6   Customer Name  9994 non-null   str           
 7   Segment        9994 non-null   str           
 8   City           9994 non-null   str           
 9   State          9994 non-null   str           
 10  Postal Code    9994 non-null   int64         
 11  Region         9994 non-null   str           
 12  Product ID     9994 non-null   str           
 13  Category       9994 non-null   str           
 14  Sub-Category   9994 non-null   str           
 15  Product Name   9994 non-null   s

In [10]:
df.describe(include='all')

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Profit Margin,Ship Delay,Order Year,Order Month,Order Day
count,9994.00,9994,9994,9994,9994,9994,9994,9994,9994,9994,9994.00,9994,9994,9994,9994,9994,9994.00,9994.00,9994.00,9994.00,9994.00,9994.00,9994.00,9994.00,9994.00
unique,NaN,5009,NaN,NaN,4,793,793,3,531,49,NaN,4,1862,3,17,1850,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,CA-2017-100111,NaN,NaN,Standard Class,WB-21850,William Brown,Consumer,New York City,California,NaN,West,OFF-PA-10001970,Office Supplies,Binders,Staple envelope,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,14,NaN,NaN,5968,37,37,5191,915,2001,NaN,3203,19,6026,1523,48,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,4997.50,NaN,2016-04-30 00:07:12.259355,2016-05-03 23:06:58.571142,NaN,NaN,NaN,NaN,NaN,NaN,55190.38,NaN,NaN,NaN,NaN,NaN,229.86,3.79,0.16,28.66,0.12,3.96,2015.72,7.81,15.47
min,1.00,NaN,2014-01-03 00:00:00,2014-01-07 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,1040.00,NaN,NaN,NaN,NaN,NaN,0.44,1.00,0.00,-6599.98,-2.75,0.00,2014.00,1.00,1.00
25%,2499.25,NaN,2015-05-23 00:00:00,2015-05-27 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,23223.00,NaN,NaN,NaN,NaN,NaN,17.28,2.00,0.00,1.73,0.07,3.00,2015.00,5.00,8.00
50%,4997.50,NaN,2016-06-26 00:00:00,2016-06-29 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,56430.50,NaN,NaN,NaN,NaN,NaN,54.49,3.00,0.20,8.67,0.27,4.00,2016.00,9.00,15.00
75%,7495.75,NaN,2017-05-14 00:00:00,2017-05-18 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,90008.00,NaN,NaN,NaN,NaN,NaN,209.94,5.00,0.20,29.36,0.36,5.00,2017.00,11.00,23.00
max,9994.00,NaN,2017-12-30 00:00:00,2018-01-05 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,99301.00,NaN,NaN,NaN,NaN,NaN,22638.48,14.00,0.80,8399.98,0.50,7.00,2017.00,12.00,31.00


## EDA et Storytelling

In [2]:
df = pd.read_csv("../data/processed/superstore_processed.csv")

---

### ***Q1 | Quelle est la santé financière globale ? (Descriptif)***

Pour débuter, je vais réaliser un état des lieux macroéconomique. 

L'objectif est de mesurer les grands indicateurs (Chiffre d'Affaires, Profit, Marge globale) et d'analyser leur évolution mensuelle pour détecter d'éventuelles anomalies ou tendances globales.

In [ ]:
total_sales = df['Sales'].sum()
total_profit = df['Profit'].sum()
marge_globale = total_profit / total_sales
total_orders = df['Order ID'].nunique()
total_customers = df['Customer ID'].nunique()

print("===== EXECUTIVE SUMMARY : Santé financière globale 2014-2017\n")
print(f"Chiffre d'Affaires Total (CA) : ${total_sales:,.0f}")
print(f"Bénéfice Net (Profit) :         ${total_profit:,.0f}")
print(f"Marge Globale (Profit Margin) : {marge_globale:.2%}")
print(f"Nombre de commandes :           {total_orders:,}")
print(f"Nombre de clients uniques :     {total_customers:,}")

===== EXECUTIVE SUMMARY : Santé financière globale 2014-2017

Chiffre d'Affaires Total (CA) : $2,297,201
Bénéfice Net (Profit) :         $286,397
Marge Globale (Profit Margin) : 12.47%
Nombre de commandes :           5,009
Nombre de clients uniques :     793


In [12]:
print(df['Order Year'].unique())

[2016 2015 2014 2017]


In [ ]:
df_trend = df.groupby(['Order Year', 'Order Month'], as_index=False)[['Sales', 'Profit']].sum()

df_trend['Période'] = pd.to_datetime(df_trend['Order Year'].astype(str) + '-' + df_trend['Order Month'].astype(str).str.zfill(2) )#+ '-' + df_trend['Order Day'].astype(str).str.zfill(2))
df_trend = df_trend.sort_values(by='Période')

fig = go.Figure()

# Le Profit
fig.add_trace(go.Bar(
    x=df_trend['Période'],
    y=df_trend['Profit'],
    name='Bénéfice Net',
    marker_color='rgba(50, 171, 96, 0.6)'
))

# Le CA
fig.add_trace(go.Scatter(
    x=df_trend['Période'],
    y=df_trend['Sales'],
    mode='lines+markers',
    name="Chiffre d'Affaires",
    line=dict(color='RoyalBlue', width=2)
))

fig.update_layout(
    title='Superstore : Croissance et Saisonnalité (CA vs Bénéfices entre 2014 et 2017)',
    xaxis_title='Année et Mois',
    yaxis_title='Montant ($)',
    hovermode='x unified',
    plot_bgcolor='white',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')

fig.show()

**Interprétation :**

1. **Une forte saisonnalité** : L'activité est portée par la fin d'année (Septembre à Décembre) correspondant à la rentrée (Back to School) et aux fêtes. Le premier trimestre (Janvier - Février) est systématiquement en creux.
2. **Le cas spécifique du mois d'Octobre (Mois de transition)** : On observe une baisse du CA en Octobre. Cependant, ce mois affiche souvent une excellente rentabilité (ex: record absolu de profit en Octobre 2016 à 16.2k$ pour "seulement" 59k$ de CA).
3. **Décorrélation ponctuelle Ventes / Profit** : 
   - En **Novembre 2017**, le CA bat son record historique (118k$), mais le profit n'est que de 9.6k$.
   - Des pertes nettes sont apparues par le passé sur des mois spécifiques (ex: -841$ en Juillet 2014, -3.2k$ en Janvier 2015).
4. **Stabilisation récente (Mi-2017)** : À partir de Mai 2017, on remarque que les profits se stabilisent (entre 6k$ et 11k$) et fluctuent beaucoup moins que les années précédentes. Cela suggère une possible correction récente de leur stratégie commerciale.

*👉 **Hypothèse 01** : "Les chutes drastiques de rentabilité (les faibles marges lors des pics de ventes) sont directement liées à des campagnes de remises (Discount) ponctuellement trop agressives. L'entreprise semble d'ailleurs avoir commencé à rationaliser cette pratique courant 2017."*

#### Vérification de l'hypothèse 01

In [ ]:
df_discount_verif = df.groupby(['Order Year', 'Order Month'], as_index=False)[['Discount']].mean()
df_discount_verif['Période'] = pd.to_datetime(df_discount_verif['Order Year'].astype(str) + '-' + df_discount_verif['Order Month'].astype(str).str.zfill(2))#+ '-' + df_trend['Order Day'].astype(str).str.zfill(2))
df_discount_verif = df_discount_verif.sort_values(by='Période')

fig2 = go.Figure()

fig2.add_trace(go.Scatter(x=df_discount_verif['Période'], y=df_discount_verif['Discount'], 
                          mode='lines+markers', name='Remise moyenne', line=dict(color='indianred', width=3)))

fig2.update_layout(
    title='Évolution du taux de remise moyen par mois (2014-2017)',
    xaxis_title='Période',
    yaxis=dict(title='Taux de Remise', tickformat='.1%'),
    plot_bgcolor='white', hovermode='x unified',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig2.update_xaxes(showgrid=True, gridcolor='LightGray')
fig2.update_yaxes(showgrid=True, gridcolor='LightGray')

fig2.show()

In [32]:
fig3 = go.Figure()

fig3.add_trace(go.Scatter(x=(df_discount_verif.groupby('Order Month', as_index=False)['Discount'].mean())['Order Month'],
                          y=(df_discount_verif.groupby('Order Month', as_index=False)['Discount'].mean())['Discount'], 
                          mode='lines+markers', name='Remise moyenne', line=dict(color='indianred', width=3)))

fig3.update_layout(
    title='Évolution du taux agrégé de remise moyen par mois',
    xaxis=dict(title='Mois', tickmode='linear', tick0=1, dtick=1),
    yaxis=dict(title='Taux de Remise', tickformat='.1%'),
    plot_bgcolor='white',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig3.update_xaxes(showgrid=True, gridcolor='LightGray')
fig3.update_yaxes(showgrid=True, gridcolor='LightGray')

fig3.show()

**Observations suite à la vérification :**

Contrairement à ce que l'on pouvait penser, la donnée nous montre que l'entreprise n'a absolument pas corrigé sa politique de remises en 2017 (fin 2017 a les mêmes taux que fin 2014). De plus, le taux agrégé de remise est étonnamment plat toute l'année (moyenne lissée autour de 15-16%).

Ainsi, ma première hypothèse est invalidée dans sa dimension temporelle globale. Le problème ne vient pas d'une augmentation générale des remises lors des pics de ventes.

*👉 **Nouvelle Hypothèse (Hypothèse 02)** : Puisque la moyenne lisse la donnée, la destruction de rentabilité, qui tire le profit total vers le bas, provient très probablement de remises ciblées et potentiellement très agressives appliquées sur des dimensions spécifiques : soit sur certaines zones géographiques (Régions/États), soit sur certaines gammes de produits.*

---

### ***Q2 | Quelles régions et états sont rentables ou déficitaires ? (Diagnostique)***

L'objectif ici est de spatialiser nos performances. Je vais analyser la rentabilité au niveau des grandes Régions d'abord, puis descendre au niveau de chaque État pour isoler géographiquement où l'entreprise crée ou détruit de la valeur.

#### Au niveau des régions

In [6]:
df_region = df.groupby('Region', as_index=False).agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Discount': 'mean'
}).sort_values(by='Profit', ascending=False)

fig4 = px.bar(
    df_region, 
    x='Region', 
    y=['Sales', 'Profit'], 
    barmode='group',
    title="Rentabilité par Région (CA vs Profit)",
    labels={'value': 'Montant ($)', 'variable': 'KPI'},
    color_discrete_map={'Sales': 'RoyalBlue', 'Profit': 'MediumSeaGreen'},
    text_auto='.2s'
)

fig4.update_layout(plot_bgcolor='white', hovermode='x unified')
fig4.update_yaxes(showgrid=True, gridcolor='LightGray')
fig4.show()

In [10]:
display(df_region.style.background_gradient(cmap='Reds', subset=['Discount']))

,Region,Sales,Profit,Discount
3,West,725457.824500,108418.448900,0.109335
1,East,678781.240000,91522.780000,0.145365
2,South,391721.905000,46749.430300,0.147253
0,Central,501239.890800,39706.362500,0.240353


*   **La région "West" (Ouest) est la championne absolue :** Elle génère le plus gros Chiffre d'Affaires (725k$) ET le plus gros bénéfice net (108k$). Son faible taux de remise moyenne *(la plus faible remise moyenne : seulement ~11%)* y est probablement pour beaucoup.
*   **Le désastre de la région "Central" :** Bien qu'elle génère un CA très correct (501k$, soit plus que le Sud), elle dégage **le pire profit (seulement 39k$)**. La raison tombe sous le sens : c'est la région qui pratique **un taux de remise désastreux à plus de 24% en moyenne**. <br>

#####

>**Ces constats semblent valider mon hypothèse 2.**

Pour en être sûr, je vais :  <br>
1. *retirer la région centrale pour voir son effet sur la marge globale de l'entreprise ;*
2. *analyser les taux de ventes à pertes dans les différentes régions.*

In [16]:
global_margin = df['Profit'].sum() / df['Sales'].sum()

df_no_central = df[df['Region'] != 'Central']
margin_no_central = df_no_central['Profit'].sum() / df_no_central['Sales'].sum()

print("===== L'IMPACT DE LA RÉGION 'CENTRAL' =====")
print(f"Marge globale actuelle (avec Central) : {global_margin:.2%}")
print(f"Marge globale simulée (SANS Central) :  {margin_no_central:.2%}")
print(f"👉 Variation : {(global_margin - margin_no_central)*100:.2f} points\n")

===== L'IMPACT DE LA RÉGION 'CENTRAL' =====
Marge globale actuelle (avec Central) : 12.47%
Marge globale simulée (SANS Central) :  13.74%
👉 Variation : -1.27 points



In [17]:
df['Is_Loss'] = df['Profit'] < 0
loss_rate_region = (df.groupby('Region')['Is_Loss'].mean() * 100).sort_values(ascending=False).to_frame("% de ventes à PERTE")

display(loss_rate_region.style.background_gradient(cmap='Reds'))

,% de ventes à PERTE
Region,
Central,31.898407
East,19.417135
South,15.987654
West,9.928192


**Le poids de la région Central** :

1. **L'impact sur la marge globale** : La première analyse montre que la région "Central" tire significativement la rentabilité vers le bas (chute de 17%). Sans elle, la marge globale de l'entreprise s'améliorerait de 1.27 point (passant de 12.47% à 13.74%). Cela représente une différence financière majeure.
2. **Une pratique systémique de ventes à perte** : Le tableau des ventes à perte révèle une information cruciale. Si la région Central affiche un taux critique (près de 32% de ses ventes se font à perte), cette pratique n'est pas une anomalie isolée. L'Est (19.4%) et le Sud (16%) affichent également des proportions très élevées de transactions déficitaires. Seul l'Ouest parvient à limiter ce taux sous la barre des 10%.

**Ce que cela nous apprend :**
L'Hypothèse 02 se précise. *Son volet géographique est vérifié en macro (au niveau des régions)*. <br> Mais la destruction de marge n'est pas *uniquement* un problème propre à la région Central. Ces chiffres suggèrent une stratégie commerciale globale (probablement liée à une politique de remises très agressive) qui accepte structurellement de vendre à perte d'importants volumes de marchandises. La région Central ne fait que pousser cette pratique à un niveau insoutenable.

#### Au niveau des États

In [ ]:
us_state_abbrev = get_us_state_abbrev()

df_state = df.groupby('State', as_index=False).agg({
    'Sales': 'sum', 
    'Profit': 'sum', 
    'Discount': 'mean'
})

df_state['State Code'] = df_state['State'].map(us_state_abbrev)

fig_map = px.choropleth(
    df_state, 
    locations='State Code', 
    locationmode="USA-states", 
    color='Profit',
    scope="usa",
    hover_name='State',
    hover_data={'State Code': False, 'Sales': ':$,.0f', 'Profit': ':$,.0f', 'Discount': ':.1%'},
    color_continuous_scale='RdBu',
    color_continuous_midpoint=0,
    title="Cartographie de la Rentabilité par État"
)
fig_map.update_layout(margin=dict(l=0, r=0, t=40, b=0))
fig_map.show()

In [22]:
from plotly.subplots import make_subplots

top_10 = df_state.sort_values(by='Profit', ascending=False).head(10)
flop_10 = df_state.sort_values(by='Profit', ascending=True).head(10)

fig_tf = make_subplots(rows=1, cols=2, subplot_titles=("Top 10 - Profits max", "Flop 10 - Pertes max"))

# Top 10
fig_tf.add_trace(go.Bar(
    x=top_10['State'], y=top_10['Profit'], 
    marker=dict(color=top_10['Discount'], colorscale='Greens', reversescale=True),
    name='Profit (Top)',
    text=[f"Remise: {d:.1%}" for d in top_10['Discount']], textposition='auto'
), row=1, col=1)

# Flop 10
fig_tf.add_trace(go.Bar(
    x=flop_10['State'], y=flop_10['Profit'], 
    marker=dict(color=flop_10['Discount'], colorscale='Reds'),
    name='Pertes (Flop)',
    text=[f"Remise: {d:.1%}" for d in flop_10['Discount']], textposition='auto'
), row=1, col=2)

fig_tf.update_layout(
    title_text="Corrélation entre les Pertes et les fortes Remises",
    showlegend=False, 
    plot_bgcolor='white',
    height=500
)
fig_tf.update_yaxes(title_text="Profit ($)", row=1, col=1)
fig_tf.show()

**Résultats de mon analyse**

1. **Forte corrélation entre pertes et remises élevées** : Les chiffres sont nets. **Tous les États du Top 10** ont une politique de remise très faible (entre 0% et 7% maximum). À l'inverse, **tous les États du Flop 10** pratiquent des remises moyennes très prononcées, avoisinant ou dépassant les 30% (jusqu'à 39% pour l'Illinois).
2. **Un volume de ventes élevé ne garantit pas la rentabilité** : L'exemple du Texas est frappant. Il génère le 3ème plus gros Chiffre d'Affaires du pays (170k$). Pourtant, avec un taux de remise moyen de 37%, il affiche le pire bilan financier avec une perte de -25,7k$. Augmenter le volume des ventes avec de tels taux de remise ne fait qu'aggraver le déficit.
3. **Une concentration des pertes (Loi de Pareto)** : Le déficit n'est pas homogène sur le territoire. Sur les 49 États étudiés, seulement 4 États (Texas, Ohio, Pennsylvanie, Illinois) cumulent à eux seuls plus de **70k$ de pertes sèches**. Ils absorbent une grande part de la rentabilité générée par des États moteurs comme New York ou la Californie.

👉 **Conclusion / Réponse à l'Hypothèse 02** : Le **volet géographique de notre hypothèse est totalement validé**. La destruction de la marge est bien corrélée à une politique de remise trop agressive, ciblée sur des zones géographiques spécifiques. Dès lors, une recommandation claire émerge : il faut urgemment revoir et limiter les plafonds de remises dans ces 4 États déficitaires.

---

#### Conclusion de la Q2 (Diagnostique Géographique)

À la question "*Quelles régions et états sont rentables ou déficitaires ?*", nous pouvons désormais répondre de manière ferme et chiffrée :

*   **Régions Performantes :** L'Ouest (West) est sans conteste le moteur de l'entreprise, avec plus de 108k$ de profit. Son "*secret*" réside dans une grande discipline sur les remises *(moyenne à ~11%)* qui limite les ventes à perte *(moins de 10% des transactions)*.
*   **Régions Déficitaires :** La région Central est en situation de naufrage (*taux de remise de 24% conduisant à ~32% de ventes à perte*).
*   **Les États "Piliers" vs "Trous Noirs" :** La rentabilité répond à la loi de Pareto. Des États extrêmement rigoureux sur les remises portent l'entreprise vers le haut (Californie, New York, Washington). À l'inverse, seulement 4 États (Texas, Ohio, Pennsylvanie, Illinois) utilisent des remises insoutenables (plus de 30%) entraînant une perte cumulée de plus de 70 000$.

Je vais donc basculer vers les dimensions produits, pour vérifier si ce problème est lié aux types de biens que l'on y vend.

---

### ***Q3 | Quelles catégories et sous-catégories créent ou détruisent de la valeur ? (Diagnostique)***

Maintenant que nous connaissons nos fragilités géographiques, je vais chercher à savoir si le problème vient de *ce* que l'on vend. L'objectif est d'identifier avec précision quelles catégories ou sous-catégories de produits ruinent la marge et lesquelles la portent financièrement.

#### Au niveau des Catégories principales

In [29]:
df_cat = df.groupby('Category', as_index=False).agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Discount': 'mean'
}).sort_values(by='Sales', ascending=False)

fig_cat = px.bar(
    df_cat, 
    x='Category', 
    y=['Sales', 'Profit'], 
    barmode='group',
    title="Rentabilité des Grandes Familles de Produits (CA vs Profit)",
    labels={'value': 'Montant ($)', 'variable': 'Indicateur'},
    color_discrete_map={'Sales': 'RoyalBlue', 'Profit': 'MediumSeaGreen'},
    text_auto='.2s'
)

fig_cat.update_layout(plot_bgcolor='white', hovermode='x unified')
fig_cat.update_yaxes(showgrid=True, gridcolor='LightGray')
fig_cat.show()

In [28]:
display(df_cat.style.background_gradient(cmap='Blues', subset=['Sales']).background_gradient(cmap='Greens', subset=['Profit']).background_gradient(cmap='Reds', subset=['Discount']))

,Category,Sales,Profit,Discount
2,Technology,836154.033000,145454.948100,0.132323
0,Furniture,741999.795300,18451.272800,0.173923
1,Office Supplies,719047.032000,122490.800800,0.157285


**Mon analyse :**

**La catégorie "Furniture" (Mobilier) est un naufrage financier.** 

1. **Le paradoxe du volume** : Elle génère un chiffre d'affaires massif (plus de 741k$), presque équivalent au pilier *Technology*, mais ne dégage qu'un profit microscopique (18,5k$). *C'est un retour sur investissement totalement inefficace.*
2. **Le vecteur du problème** : Sans aucune surprise, c'est encore une fois la catégorie qui subit le taux de remise moyen le plus élevé (près de 17.4% lissés sur toute l'année et tout le pays).
3. **Les moteurs de rentabilité** : À l'inverse, *Technology* et *Office Supplies* sont les vrais piliers économiques de l'enseigne. Elles portent le groupe grâce à des marges saines et des remises mieux maîtrisées.

👉 *Prochaine étape* : Je vais "ouvrir le capot" de ces catégories et descendre au niveau des **Sous-catégories**.<br> Mon objectif ici est double : trouver quels types de meubles exacts ruinent la catégorie *Furniture*, et vérifier s'il ne se cache pas des produits "toxiques" dans les autres catégories.

#### Au niveau des Sous-Catégories

In [31]:
df_subcat = df.groupby(['Category', 'Sub-Category'], as_index=False).agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Discount': 'mean'
}).sort_values(by='Profit', ascending=True)

fig_subcat = px.bar(
    df_subcat, 
    y='Sub-Category', 
    x='Profit', 
    orientation='h',
    color='Discount',
    color_continuous_scale='Reds',
    text_auto='$.2s',
    hover_data={'Category': True, 'Sales': ':$,.0f', 'Discount': ':.1%'},
    title="Profit par Sous-Catégorie organisé par niveau de Remise",
    labels={'Sub-Category': '', 'Profit': 'Profit ($)'}
)

fig_subcat.update_layout(
    plot_bgcolor='white', 
    height=600,
    hovermode='y unified'
)

fig_subcat.update_xaxes(showgrid=True, gridcolor='LightGray')
fig_subcat.show()

**Mon analyse :**

1. **La contre-performance "Furniture" est très localisée** : Le problème de la catégorie Mobilier ne provient pas de tous ses produits. Les **Tables** constituent le plus gros gouffre financier de l'entreprise (-17,7k$ de déficit net), entraînées par un taux de remise moyen lourd (26%). Les **Bookcases** (Bibliothèques) suivent la même mécanique destructrice (-3,4k$ avec 21% de remise). À l'inverse, les **Chairs** (Chaises) et les **Furnishings** restent relativement rentables.
2. **Le paradoxe des "Binders"** : Il s'agit du produit qui subit *le plus fort taux de remise moyen* de tout le catalogue (37%). Pourtant, il génère le 5ème plus gros profit global (+30,2k$). Cela démontre que ces articles possèdent une marge brute initiale immense, capable d'absorber des stratégies promotionnelles agressives.
3. **Les "Copiers" sont la vache à lait** : Ils génèrent le profit absolu le plus élevé (+55,6k$) avec un volume de ventes modéré (149k$) et un discount maîtrisé (16%). C'est le produit le plus efficace financièrement du catalogue.
4. **Surveillance requise sur "Machines"** : Au sein de la catégorie très rentable "Technology", les "Machines" affichent un ratio inquiétant : 189k$ de CA pour seulement 3,3k$ de profit. Probablement que c'est principalement en raison de remises très élevées (31%).

👉 **Conclusion / Réponse à l'Hypothèse 02** : Le **second volet de mon hypothèse est lui aussi validé**. La destruction de la rentabilité n'est pas globale, elle est bien causée par des taux de remises agressifs concentrés sur des sous-catégories spécifiques (comme les Tables et les Machines logées au milieu de catégories pourtant saines). L'entreprise saigne localement et non globalement.

---

#### Conclusion de la Q3

La stratégie de remise n'est pas "mauvaise" par nature (comme le prouve le succès des Binders), mais elle est appliquée de manière aveugle sur des produits dont la structure de coût ne peut pas la supporter (comme les Tables et les Machines). 

Pour corriger cela, je vais essayer de calculer mathématiquement le "Point Mort" (Break-Even Point) pour proposer un plafond de remise maximal acceptable. C'est l'objet de la prochaine analyse (Q4).

---

### ***Q4 | Les remises sont-elles une stratégie rentable ou un problème structurel ? (Diagnostique)***

Nos analyses précédentes pointent toutes vers le rôle destructeur et incontrôlé des soldes. L'objectif de cette section est de calculer mathématiquement le "Point Mort" (Break-Even Point), c'est-à-dire le taux de remise précis à partir duquel l'entreprise commence systématiquement à vendre à perte.

In [36]:
df_discount = df.groupby('Discount', as_index=False).agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Order ID': 'count'
}).rename(columns={'Order ID': 'Volume de Transactions'})

df_discount['Marge Bénéficiaire'] = df_discount['Profit'] / df_discount['Sales']
df_discount['Profit Moyen Par Transaction'] = df_discount['Profit'] / df_discount['Volume de Transactions']

fig_be = go.Figure()

fig_be.add_trace(go.Scatter(
    x=df_discount['Discount'], 
    y=df_discount['Marge Bénéficiaire'],
    mode='lines+markers',
    name='Marge Bénéficiaire',
    line=dict(color='royalblue', width=3),
    marker=dict(size=8, color=df_discount['Marge Bénéficiaire'], colorscale='RdYlGn', showscale=False)
))

fig_be.add_hline(y=0, line_dash="dash", line_color="red", 
                 annotation_text=" Point Mort de Rentabilité (Marge = 0%)",
                 annotation_position="bottom right")

fig_be.update_layout(
    title="La Chute Gravitationnelle de la Marge selon le Taux de Remise",
    xaxis=dict(title='Taux de Remise', tickformat='.0%'),
    yaxis=dict(title='Marge Bénéficiaire Nette', tickformat='.0%'),
    plot_bgcolor='white',
    hovermode='x unified',
    height=500
)

fig_be.update_xaxes(showgrid=True, gridcolor='LightGray')
fig_be.update_yaxes(showgrid=True, gridcolor='LightGray')

fig_be.show()

In [37]:
display(df_discount.style.background_gradient(cmap='RdYlGn', subset=['Marge Bénéficiaire', 'Profit Moyen Par Transaction']).format({
    'Discount': '{:.0%}',
    'Sales': '${:,.0f}',
    'Profit': '${:,.0f}',
    'Marge Bénéficiaire': '{:.1%}',
    'Profit Moyen Par Transaction': '${:,.2f}'
}))

,Discount,Sales,Profit,Volume de Transactions,Marge Bénéficiaire,Profit Moyen Par Transaction
0,0%,"$1,087,908","$320,988",4798,29.5%,$66.90
1,10%,"$54,369","$9,029",94,16.6%,$96.06
2,15%,"$27,559","$1,419",52,5.1%,$27.29
3,20%,"$764,594","$90,337",3657,11.8%,$24.70
4,30%,"$103,227","$-10,369",227,-10.0%,$-45.68
5,32%,"$14,493","$-2,391",27,-16.5%,$-88.56
6,40%,"$116,418","$-23,057",206,-19.8%,$-111.93
7,45%,"$5,485","$-2,493",11,-45.5%,$-226.65
8,50%,"$58,919","$-20,506",66,-34.8%,$-310.70
9,60%,"$6,645","$-5,945",138,-89.5%,$-43.08


**Observations**

1. **Le seuil de rentabilité (Break-Even) est franchi à 30% de remise :** Jusqu'à 20% de réduction, l'entreprise gagne de l'argent. Dès que la remise atteint **30%**, la transaction devient systématiquement déficitaire (marge nette de -10%, soit une perte d'environ 45$ par transaction). Plus la remise augmente ensuite, plus le gouffre se creuse (jusqu'à -180% de marge à 80% de discount).
2. **Le palier optimal des 20% :** Ce palier est fascinant. Il concentre un volume énorme (3 657 transactions et plus de 764k$ de CA), tout en maintenant une marge très saine (11,8%). C'est la preuve que l'entreprise sait faire des promotions rentables. Ce palier de 20% semble être la limite maximale d'optimisation volume/profit pour la grande majorité du catalogue.
3. **Le fléau des taux "toxiques" (≥ 30%) :**  En cumulant les lignes de 30% à 80%, on observe que l'entreprise a réalisé près de **1 380 transactions** avec des taux destructeurs de valeur. Ce sont précisément ces transactions qui expliquent les pertes massives constatées dans les États comme le Texas (où la moyenne des remises est de 37%) ou sur le Mobilier (où les Tables sont à 26% de moyenne, ce qui implique que beaucoup d'entre elles dépassent le seuil des 30%).

---

#### Conclusion de la Q4
Le diagnostic est clair. Le problème n'est pas le principe de la remise, mais l'absence de garde-fous.   
>**Recommandation :** Implémenter une règle stricte bloquant systémiquement toute remise supérieure à 20%, qui s'avère être le palier maximal de rentabilité (avec d'éventuelles exceptions très encadrées pour des produits à très haute marge brute comme les classeurs).

---

### ***Q5 | Quel profil client est le plus rentable, et pourquoi ? (Prescriptif)***

Pour la dernière dimension analytique, je vais auditer les **Segments** clients : *Consumer* (B2C), *Corporate* (B2B classique) et *Home Office* (TPE/Indépendants).

L'objectif est d'identifier non seulement qui achète le plus, mais surtout **qui est le plus rentable individuellement** (le Profit généré par Client unique) afin de cibler nos futures campagnes d'acquisition.

In [39]:
df_segment = df.groupby('Segment', as_index=False).agg({
    'Sales': 'sum',
    'Profit': 'sum',
    'Discount': 'mean',
    'Order ID': 'nunique',
    'Customer ID': 'nunique'
}).rename(columns={'Order ID': 'Commandes', 'Customer ID': 'Clients Uniques'})

df_segment['Panier Moyen (CA)'] = df_segment['Sales'] / df_segment['Commandes']
df_segment['Panier Moyen (Profit)'] = df_segment['Profit'] / df_segment['Commandes']
df_segment['Valeur par Client (Profit/Client)'] = df_segment['Profit'] / df_segment['Clients Uniques']
df_segment['Marge'] = df_segment['Profit'] / df_segment['Sales']

df_segment = df_segment.sort_values(by='Valeur par Client (Profit/Client)', ascending=False)

fig_seg = go.Figure()

fig_seg.add_trace(go.Bar(
    x=df_segment['Segment'], 
    y=df_segment['Valeur par Client (Profit/Client)'],
    name='Profit par Client Unique ($)',
    marker_color='MediumSeaGreen',
    text=[f"${val:,.0f}" for val in df_segment['Valeur par Client (Profit/Client)']],
    textposition='auto'
))

fig_seg.update_layout(
    title='Profit par Client Unique selon son Segment',
    xaxis_title='Segment Client',
    yaxis_title='Profit par Client ($)',
    plot_bgcolor='white',
    hovermode='x unified',
    height=450
)

fig_seg.update_yaxes(showgrid=True, gridcolor='LightGray')

fig_seg.show()

In [40]:
display(df_segment.style.background_gradient(cmap='Greens', subset=['Valeur par Client (Profit/Client)', 'Marge']).format({
    'Sales': '${:,.0f}',
    'Profit': '${:,.0f}',
    'Discount': '{:.1%}',
    'Panier Moyen (CA)': '${:,.0f}',
    'Panier Moyen (Profit)': '${:,.0f}',
    'Valeur par Client (Profit/Client)': '${:,.0f}',
    'Marge': '{:.1%}'
}))

,Segment,Sales,Profit,Discount,Commandes,Clients Uniques,Panier Moyen (CA),Panier Moyen (Profit),Valeur par Client (Profit/Client),Marge
2,Home Office,"$429,653","$60,299",14.7%,909,148,$473,$66,$407,14.0%
1,Corporate,"$706,146","$91,979",15.8%,1514,236,$466,$61,$390,13.0%
0,Consumer,"$1,161,401","$134,119",15.8%,2586,409,$449,$52,$328,11.5%


**Observations :**

1. **Le paradoxe du volume vs rentabilité (Consumer)** : Le segment *Consumer* (B2C) écrase les autres en valeur absolue (1,16M$ de CA pour 409 clients). Cependant, c'est une illusion de volume. C'est le segment avec la marge la plus faible (11.5%) et la plus petite rentabilité par individu (328$ de profit généré par client).
2. **Le joyau caché "Home Office"** : Bien qu'il représente la plus petite part du portefeuille (seulement 148 clients uniques), ce segment est de loin le plus performant à l'échelle individuelle. Chaque client *Home Office* rapporte en moyenne **407$ de bénéfice net** à l'entreprise, en affichant la meilleure marge globale (14.0%).
3. **L'explication par le comportement d'achat** : Ce n'est pas un hasard si le segment *Home Office* est le plus rentable. Le tableau montre qu'ils subissent le taux de remise moyen le plus bas de l'entreprise (14.7%) tout en dégageant le panier moyen par commande le plus élevé (473$). Leur comportement est mécaniquement plus sain financièrement (plus d'équipements, moins de chasse pure à la promotion).


---

#### Conclusion de la Q5

Le segment B2C (*Consumer*) fait tourner l'entrepôt, mais c'est le segment *Home Office* qui enrichit réellement l'entreprise proportionnellement à l'effort fourni. 

>**Recommandation :** En termes de Coût d'Acquisition Client (CAC), l'entreprise a tout intérêt à réorienter et augmenter son budget marketing/commercial pour cibler spécifiquement les TPE/Indépendants (*Home Office*). Attirer un client dans ce segment s'avère **24% plus rentable** que d'acquérir un client classique (Consumer). Il faut concevoir des offres d'équipements (notamment de Technologie, très rentable) dédiées aux travailleurs à domicile.